# Baselines B0–B3 – Lê Quốc Đại

Content Moderation using MGTAB

Models:
- B0: Majority
- B1: Content-only
- B2a: Graph-only
- B3: Label Propagation

Evaluation: Shared train/validation split, validation metrics only.

In [ ]:
# 1. Setup (Colab or local)
import os, sys
if not os.path.exists("mgtab.py"):
    if "google.colab" in sys.modules:
        if not os.path.exists("sna-content-moderation"):
            !git clone -q https://github.com/ThSonDev/sna-content-moderation.git
        %cd sna-content-moderation
        !pip install -q -r requirements.txt
    else:
        %cd ..
sys.path.insert(0, os.getcwd())

/content/sna-content-moderation


In [ ]:
# 2. Download MGTAB (31 MB) into data/MGTAB, only once
import urllib.request, zipfile
if not os.path.exists("data/MGTAB/features.pt"):
    os.makedirs("data", exist_ok=True)
    url = "https://drive.usercontent.google.com/download?id=1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe&export=download&confirm=t"
    urllib.request.urlretrieve(url, "data/MGTAB.zip")
    zipfile.ZipFile("data/MGTAB.zip").extractall("data")

In [ ]:
# 3. Load data and the shared split (never make your own split)
import numpy as np
from mgtab import load, relations, split, best_threshold, evaluate, RELATIONS
X, y, stance, E, T, W = load()
rel = relations(E, T, len(y))   # {"follow": (src, dst), ...}
train, val, test = split(y)     # test stays locked until the final run
print(X.shape, {r: len(s) for r, (s, d) in rel.items()})

(10199, 788) {'follow': 425863, 'mention': 110868, 'reply': 22327, 'quote': 20901, 'url': 263800, 'hashtag': 300000}


## Your code

Rules: fit scalers and models on `train` only; report `val` only; tune the threshold with `best_threshold` on `val`;
never use `stance` as a feature. Save result tables to `results/<task>.csv`.

In [ ]:

# ============================================================
# BASELINES - SNA CONTENT MODERATION (MGTAB)
# Owner: Dai
# B0 Majority | B1 Content-only | B2a Graph-only | B3 LP
# Evaluation: validation only; test labels remain unused
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

from features import struct_features, PROFILE_ONLY

os.makedirs("results", exist_ok=True)

assert X.shape == (10199, 788)
assert len(train) == 7139 and len(val) == 1530

results = {}

# ------------------------------------------------------------
# Common LR: same configuration as P1
# ------------------------------------------------------------
def run_lr(Z):
    best = None

    for C in [0.01, 0.1, 1, 10]:
        clf = LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=5000
        )
        clf.fit(Z[train], y[train])

        # Only validation predictions are needed for evaluation.
        scores = clf.predict_proba(Z[val])[:, 1]
        ap = average_precision_score(y[val], scores)

        if best is None or ap > best[0]:
            best = (ap, C, scores)

    _, C, scores = best
    threshold = best_threshold(y[val], scores)

    return {
        "C": C,
        **evaluate(y[val], scores, threshold)
    }


# ------------------------------------------------------------
# B0: Majority - always predicts human
# ------------------------------------------------------------
print("Running B0: Majority...")

scores_b0 = np.zeros(len(val), dtype=float)
results["B0 Majority"] = {
    "C": np.nan,
    **evaluate(y[val], scores_b0, 0.5)
}

print("B0 completed.")


# ------------------------------------------------------------
# B1: Content-only - P (16) + T (768)
# ------------------------------------------------------------
print("\nRunning B1: Content-only...")

P = X[:, PROFILE_ONLY]
Tw = X[:, 20:]

Z_content = np.hstack([P, Tw])

assert Z_content.shape == (10199, 784)

results["B1 Content-only"] = run_lr(Z_content)

print("B1 completed.")
print("B1 PR-AUC:", results["B1 Content-only"]["pr_auc"])


# ------------------------------------------------------------
# B2a: Graph-only - G-struct (32)
# ------------------------------------------------------------
print("\nRunning B2a: Graph-only...")

G, g_names = struct_features(X, rel)

assert G.shape == (10199, 32)
assert np.isfinite(G).all()

# Fit scaler on training accounts only.
scaler = StandardScaler().fit(G[train])
G_scaled = scaler.transform(G)

results["B2a Graph-only"] = run_lr(G_scaled)

print("B2a completed.")
print("B2a PR-AUC:", results["B2a Graph-only"]["pr_auc"])


# ------------------------------------------------------------
# B3: Label Propagation - merged undirected graph
# ------------------------------------------------------------
print("\nRunning B3: Label Propagation...")

n = len(y)

# Build undirected graph from all six relations.
src_parts = []
dst_parts = []

for relation_name, (src, dst) in rel.items():
    src_parts.extend([src, dst])
    dst_parts.extend([dst, src])

src_all = np.concatenate(src_parts)
dst_all = np.concatenate(dst_parts)

A = sparse.coo_matrix(
    (
        np.ones(len(src_all), dtype=np.float32),
        (src_all, dst_all)
    ),
    shape=(n, n)
).tocsr()

# Merge duplicate edges across relations.
A.sum_duplicates()
A.data[:] = 1.0

# Remove self-loops.
A.setdiag(0)
A.eliminate_zeros()

degree = np.asarray(A.sum(axis=1)).ravel()
isolated = degree == 0

# Row-normalized adjacency for neighbor averaging.
inv_degree = np.zeros(n, dtype=np.float64)
inv_degree[~isolated] = 1.0 / degree[~isolated]

transition = sparse.diags(inv_degree) @ A

# Bot prior computed only from training labels.
prior = float(y[train].mean())

# Unknown accounts start at the training bot prior.
scores_lp = np.full(n, prior, dtype=np.float64)

# Training labels are observed and fixed.
scores_lp[train] = y[train].astype(np.float64)

# Propagate until convergence or 100 iterations.
for iteration in range(100):
    new_scores = np.asarray(
        transition @ scores_lp
    ).ravel()

    # No neighbors => use the train bot proportion.
    new_scores[isolated] = prior

    # Clamp known training labels.
    new_scores[train] = y[train]

    diff = np.max(np.abs(new_scores - scores_lp))
    scores_lp = new_scores

    if diff < 1e-6:
        break

scores_b3 = scores_lp[val]
threshold_b3 = best_threshold(y[val], scores_b3)

results["B3 Label Propagation"] = {
    "C": np.nan,
    **evaluate(y[val], scores_b3, threshold_b3)
}

print("B3 completed.")
print("Iterations:", iteration + 1)
print("Isolated accounts:", int(isolated.sum()))
print("B3 PR-AUC:", results["B3 Label Propagation"]["pr_auc"])


# ------------------------------------------------------------
# Save and compare with P1
# ------------------------------------------------------------
columns = [
    "C", "pr_auc", "roc_auc",
    "macro_f1", "p@100", "p@200"
]

res = pd.DataFrame.from_dict(
    results, orient="index"
)[columns]

res.to_csv("results/baselines.csv")

print("\n========== BASELINE RESULTS ==========")
display(res.round(4))

print("\nSaved: results/baselines.csv")

# B1 should reproduce P1 without G-struct.
reference_pr_auc = 0.8212172626804594
difference = abs(
    results["B1 Content-only"]["pr_auc"] - reference_pr_auc
)

print("\nB1 vs P1 without G-struct:")
print("Absolute PR-AUC difference:", difference)

if difference < 0.001:
    print("PASS: B1 agrees with the P1 reference.")
else:
    print("CHECK: B1 differs from P1; review preprocessing and LR.")


Running B0: Majority...
B0 completed.

Running B1: Content-only...
B1 completed.
B1 PR-AUC: 0.8211401812922816

Running B2a: Graph-only...
B2a completed.
B2a PR-AUC: 0.7593823908846333

Running B3: Label Propagation...
B3 completed.
Iterations: 12
Isolated accounts: 54
B3 PR-AUC: 0.5283486212419857

========== BASELINE RESULTS ==========


,C,pr_auc,roc_auc,macro_f1,p@100,p@200
B0 Majority,NaN,0.2693,0.5000,0.4222,0.22,0.22
B1 Content-only,0.1,0.8211,0.9302,0.8405,0.92,0.89
B2a Graph-only,0.1,0.7594,0.9251,0.8340,0.80,0.82
B3 Label Propagation,NaN,0.5283,0.7040,0.6845,0.69,0.63



Saved: results/baselines.csv

B1 vs P1 without G-struct:
Absolute PR-AUC difference: 7.708138817774657e-05
PASS: B1 agrees with the P1 reference.


B0 assigns identical scores to all accounts. Its Precision@100 and Precision@200 depend on tie ordering and do not represent a meaningful ranking.

In [ ]:
from google.colab import files
files.download("results/baselines.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>